# 🌌 Interactive Galactic Civilization Simulation

Explore the dynamics of intelligent civilizations spreading through a Milky Way-like galaxy with interactive controls and realistic 3D visualizations.

## Features:
- **Interactive parameter control** via widgets
- **Real-time 3D visualizations** using Plotly
- **Live simulation** with progress tracking
- **Exploration mode** for analyzing existing runs
- **Realistic astrophysical rendering**

In [ ]:
# Setup and imports
import sys
from pathlib import Path

# Add project to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import json
from datetime import datetime

# Interactive widgets
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Simulation
from examples.realistic_3d_simulation import Realistic3DSimulation
from examples.deadly_simulation import DeadlySimulation

print("✅ All imports successful!")

## 🎛️ Simulation Control Panel

Configure simulation parameters and run settings.

In [ ]:
# Create control widgets
style = {'description_width': '200px'}
layout = widgets.Layout(width='500px')

# Simulation mode
mode_selector = widgets.RadioButtons(
    options=['Realistic (with cooperation)', 'DEADLY (no cooperation, 100x hazards)'],
    value='Realistic (with cooperation)',
    description='Mode:',
    style=style
)

# Basic parameters
num_civs_slider = widgets.IntSlider(
    value=20,
    min=5,
    max=100,
    step=5,
    description='Number of Civilizations:',
    style=style,
    layout=layout
)

num_stars_slider = widgets.IntSlider(
    value=50000,
    min=10000,
    max=200000,
    step=10000,
    description='Number of Stars:',
    style=style,
    layout=layout
)

duration_slider = widgets.IntSlider(
    value=5000,
    min=1000,
    max=20000,
    step=1000,
    description='Duration (Myr):',
    style=style,
    layout=layout
)

seed_input = widgets.IntText(
    value=42,
    description='Random Seed:',
    style=style,
    layout=layout
)

# Run button
run_button = widgets.Button(
    description='🚀 Run Simulation',
    button_style='success',
    layout=widgets.Layout(width='200px', height='40px')
)

# Output area
output_area = widgets.Output()

# Display controls
control_panel = widgets.VBox([
    widgets.HTML("<h3>⚙️ Simulation Parameters</h3>"),
    mode_selector,
    num_civs_slider,
    num_stars_slider,
    duration_slider,
    seed_input,
    run_button,
    output_area
])

display(control_panel)

## 🎨 Visualization Functions

Realistic 3D rendering with Plotly.

In [ ]:
# Import visualization functions from library
from great_silence.visualization import (
    create_3d_galaxy_view,
    create_development_timeline,
    create_statistics_dashboard
)

print("✅ Visualization functions imported from library!")

## 🚀 Run Simulation

Execute simulation with selected parameters.

In [ ]:
# Global variable to store simulation data
current_sim_data = None

def run_simulation(button):
    """Run simulation with current parameters."""
    global current_sim_data
    
    with output_area:
        clear_output()
        
        # Get parameters
        mode = mode_selector.value
        num_civs = num_civs_slider.value
        num_stars = num_stars_slider.value
        duration = duration_slider.value
        seed = seed_input.value
        
        print(f"🚀 Starting {mode} simulation...")
        print(f"   Civilizations: {num_civs}")
        print(f"   Stars: {num_stars:,}")
        print(f"   Duration: {duration:,} Myr")
        print(f"   Seed: {seed}")
        print()
        
        try:
            # Create simulation
            if 'DEADLY' in mode:
                sim = DeadlySimulation(
                    num_stars=num_stars,
                    num_civilizations=num_civs,
                    seed=seed
                )
            else:
                sim = Realistic3DSimulation(
                    num_stars=num_stars,
                    num_civilizations=num_civs,
                    seed=seed
                )
            
            print("Initializing galaxy...")
            sim.initialize()
            
            print(f"Running simulation for {duration} Myr...")
            sim.run(duration_myr=float(duration), print_interval_myr=duration/10)
            
            print("\n✅ Simulation complete!")
            
            # Export data
            print("Exporting data...")
            output_path = project_root / "notebooks" / "temp_sim_data.json"
            sim.export_3d_data(str(output_path))
            
            # Load data
            with open(output_path, 'r') as f:
                current_sim_data = json.load(f)
            
            # Summary
            num_alive = current_sim_data['metadata']['num_alive']
            num_dead = current_sim_data['metadata']['num_dead']
            print(f"\n📊 Results:")
            print(f"   Survived: {num_alive}/{num_civs} ({num_alive/num_civs*100:.1f}%)")
            print(f"   Died: {num_dead}/{num_civs} ({num_dead/num_civs*100:.1f}%)")
            print(f"   Supernovae: {len(current_sim_data.get('supernova_events', []))}")
            print(f"   GRBs: {len(current_sim_data.get('grb_events', []))}")
            
            print("\n✅ Data ready for visualization! Run the visualization cells below.")
            
        except Exception as e:
            print(f"\n❌ Error: {e}")
            import traceback
            traceback.print_exc()

# Connect button
run_button.on_click(run_simulation)

print("✅ Simulation runner ready! Click the button above to start.")

## 📊 Visualize Results

Interactive 3D galaxy view with all civilizations.

In [ ]:
# 3D Galaxy View
if current_sim_data is not None:
    fig = create_3d_galaxy_view(
        current_sim_data,
        title=f"Galactic Civilization Map ({current_sim_data['metadata']['duration_myr']:.0f} Myr)"
    )
    fig.show()
else:
    print("⚠️  No simulation data. Run a simulation first!")

## 📈 Development Timelines

Track how civilizations evolved over time.

In [ ]:
# Timeline plot
if current_sim_data is not None:
    fig = create_development_timeline(current_sim_data)
    fig.show()
else:
    print("⚠️  No simulation data. Run a simulation first!")

## 📊 Statistics Dashboard

Comprehensive overview of simulation outcomes.

In [ ]:
# Statistics dashboard
if current_sim_data is not None:
    fig = create_statistics_dashboard(current_sim_data)
    fig.show()
else:
    print("⚠️  No simulation data. Run a simulation first!")

## 🔍 Load Existing Simulation

Explore previously run simulations.

In [ ]:
# File browser for existing simulations
output_dir = project_root / "outputs"

# Get all run directories
if output_dir.exists():
    runs = sorted([d.name for d in output_dir.iterdir() if d.is_dir()], reverse=True)
    
    run_selector = widgets.Dropdown(
        options=runs,
        description='Select Run:',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px')
    )
    
    load_button = widgets.Button(
        description='📂 Load Simulation',
        button_style='info',
        layout=widgets.Layout(width='200px')
    )
    
    load_output = widgets.Output()
    
    def load_existing(button):
        global current_sim_data
        with load_output:
            clear_output()
            run_name = run_selector.value
            data_path = output_dir / run_name / "data" / "simulation_3d_data.json"
            
            if data_path.exists():
                with open(data_path, 'r') as f:
                    current_sim_data = json.load(f)
                print(f"✅ Loaded: {run_name}")
                print(f"   Duration: {current_sim_data['metadata']['duration_myr']:.0f} Myr")
                print(f"   Civilizations: {current_sim_data['metadata']['num_civilizations']}")
                print(f"   Alive: {current_sim_data['metadata']['num_alive']}")
                print(f"   Dead: {current_sim_data['metadata']['num_dead']}")
                print("\n✅ Data loaded! Run visualization cells above.")
            else:
                print(f"❌ Data file not found: {data_path}")
    
    load_button.on_click(load_existing)
    
    display(widgets.VBox([
        widgets.HTML("<h3>📂 Load Existing Simulation</h3>"),
        run_selector,
        load_button,
        load_output
    ]))
else:
    print("No output directory found. Run a simulation first!")

## 🎯 Exploration Tools

Interactive filters and analysis.

In [ ]:
# Interactive exploration widgets
if current_sim_data is not None:
    # Filter by status
    status_filter = widgets.SelectMultiple(
        options=['Alive', 'Dead (any)', 'Dead (crisis)', 'Dead (supernova)', 'Dead (GRB)'],
        value=['Alive'],
        description='Show:',
        style={'description_width': '100px'},
        layout=widgets.Layout(width='400px')
    )
    
    # Kardashev range filter
    k_range = widgets.FloatRangeSlider(
        value=[0.5, 3.0],
        min=0.5,
        max=15.0,
        step=0.1,
        description='K-level Range:',
        style={'description_width': '150px'},
        layout=widgets.Layout(width='500px')
    )
    
    apply_filter_btn = widgets.Button(
        description='Apply Filters',
        button_style='primary'
    )
    
    filter_output = widgets.Output()
    
    def apply_filters(button):
        with filter_output:
            clear_output()
            
            # Filter civilizations
            filtered_civs = []
            for civ in current_sim_data['civilizations']:
                # Status filter
                if 'Alive' in status_filter.value and civ['alive']:
                    filtered_civs.append(civ)
                elif 'Dead (any)' in status_filter.value and not civ['alive']:
                    filtered_civs.append(civ)
                elif not civ['alive']:
                    cause = civ.get('cause_of_death', '')
                    if 'Dead (crisis)' in status_filter.value and any(c in cause for c in ['nuclear', 'climate', 'ai']):
                        filtered_civs.append(civ)
                    elif 'Dead (supernova)' in status_filter.value and 'supernova' in cause:
                        filtered_civs.append(civ)
                    elif 'Dead (GRB)' in status_filter.value and 'gamma' in cause:
                        filtered_civs.append(civ)
            
            # Kardashev filter
            k_min, k_max = k_range.value
            filtered_civs = [c for c in filtered_civs if k_min <= c['kardashev_level'] <= k_max]
            
            print(f"Filtered results: {len(filtered_civs)} civilizations")
            
            # Create filtered visualization
            filtered_data = current_sim_data.copy()
            filtered_data['civilizations'] = filtered_civs
            
            fig = create_3d_galaxy_view(
                filtered_data,
                title=f"Filtered View ({len(filtered_civs)} civilizations)"
            )
            fig.show()
    
    apply_filter_btn.on_click(apply_filters)
    
    display(widgets.VBox([
        widgets.HTML("<h3>🔍 Filter Civilizations</h3>"),
        status_filter,
        k_range,
        apply_filter_btn,
        filter_output
    ]))
else:
    print("⚠️  No simulation data. Run a simulation first!")

## 💾 Export Results

Save current visualization as HTML for sharing.

In [ ]:
if current_sim_data is not None:
    export_button = widgets.Button(
        description='💾 Export Interactive Plot',
        button_style='warning'
    )
    
    export_output = widgets.Output()
    
    def export_plot(button):
        with export_output:
            clear_output()
            fig = create_3d_galaxy_view(current_sim_data)
            
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            output_path = project_root / "notebooks" / f"galaxy_viz_{timestamp}.html"
            
            fig.write_html(str(output_path))
            print(f"✅ Exported to: {output_path}")
            print(f"   Open in browser for interactive 3D exploration!")
    
    export_button.on_click(export_plot)
    
    display(widgets.VBox([export_button, export_output]))
else:
    print("⚠️  No simulation data. Run a simulation first!")